In [2]:
import openeo
import rasterio
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [3]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

# Download of S2 imagery (2021) for training and testing the model

In [ ]:
# training region01
train_area_01 = {"west": 22.4, "south": 62.2, "east": 22.8, "north": 62.5}

train_cube_01 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=train_area_01,
    temporal_extent=["2021-06-01", "2021-08-31"],
    bands=["B02", "B03", "B04", "B08", "B11", "B12"],
    max_cloud_cover=30,
)

train_composite_01 = train_cube_01.reduce_dimension(dimension="t", reducer="median")

train_composite_01.download("../data/raw/raster/train_region01_2021.tif")

In [5]:
# training region02
train_area_02 = {"west": 21.8, "south": 62.4, "east": 22.2, "north": 62.7}

train_cube_02 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=train_area_02,
    temporal_extent=["2021-06-01", "2021-08-31"],
    bands=["B02", "B03", "B04", "B08", "B11", "B12"],
    max_cloud_cover=30,
)

train_composite_02 = train_cube_02.reduce_dimension(dimension="t", reducer="median")

train_composite_02.download("../data/raw/raster/train_region02_2021.tif")

In [ ]:
##### check if i need this layer?? i don think so?
# download NDVI for the training region, computed server side
train_ndvi = (train_composite.band("B08") - train_composite.band("B04")) / (train_composite.band("B08") + train_composite.band("B04"))

train_ndvi.download("../data/raw/raster/train_ndvi_2021.tif")

In [ ]:
# test region
test_area_01 = {"west": 23.2, "south": 62.2, "east": 23.6, "north": 62.5}

test_cube_01 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=test_area_01,
    temporal_extent=["2021-06-01", "2021-08-31"],
    bands=["B02", "B03", "B04", "B08", "B11", "B12"],
    max_cloud_cover=30,
)

test_composite_01 = test_cube_01.reduce_dimension(dimension="t", reducer="median")

test_composite_01.download("../data/raw/raster/test_region01_2021.tif")

# Cleaning the data

In [4]:
# clean the nodata border
with rasterio.open("../data/raw/raster/train_region01_2021.tif") as src:
    data = src.read()
    profile = src.profile
data[data == -32768] = 0
profile.update(nodata=None)

with rasterio.open("../data/processed/train_images/train_region01_2021_clean.tif", "w", **profile) as dst:
    dst.write(data)

In [5]:
# clean the nodata border
with rasterio.open("../data/raw/raster/train_region02_2021.tif") as src:
    data = src.read()
    profile = src.profile
data[data == -32768] = 0
profile.update(nodata=None)

with rasterio.open("../data/processed/train_images/train_region02_2021_clean.tif", "w", **profile) as dst:
    dst.write(data)

In [5]:
# clean the nodata border
with rasterio.open("../data/raw/raster/test_region01_2021.tif") as src:
    data = src.read()
    profile = src.profile
data[data == -32768] = 0
profile.update(nodata=None)
with rasterio.open("../data/processed/test_region01_2021_clean.tif", "w", **profile) as dst:
    dst.write(data)